# 04 — Rejection-sampling (RFT) data for GSM8K

Sample `n_samples` answers per **train / val** problem from the **base** Qwen2.5-0.5B-Instruct, keep the correct, non-truncated ones,
dedupe by their equations, and save them in the same format as `gsm8k_train.jsonl` so `02_sft` can train on them by changing only the file path.
Test files are never read.

Input: output of `00_data`. Set `DEBUG = True` first (16 problems, 2 samples), then `False`.
Resumable: each GPU appends its results to `shards/` every `flush_every` batches; rerunning the sampling cell skips finished problems.

In [1]:
DEBUG = False

In [2]:
import json, logging, re, sys, time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

CFG = dict(
    run_name="exp4_rft_data", model_id="Qwen/Qwen2.5-0.5B-Instruct", seed=42, debug=DEBUG,
    n_samples=2 if DEBUG else 8, temperature=0.7, max_new_tokens=512,
    prompts_per_batch=32*2,          # x n_samples sequences per generate call
    flush_every=10,                # batches between shard writes (resume granularity)
    max_kept_per_problem=4,
    limit=150 if DEBUG else None,   # problems per split
)
DATA_DIR = Path("/kaggle/input/notebooks/tensorpanda231/00-data-math-reasoning/data")
OUT_DIR = Path("/kaggle/working") / (CFG["run_name"] + ("_debug" if DEBUG else ""))
assert (DATA_DIR / "gsm8k_train.jsonl").exists(), f"add the 00_data notebook as an input ({DATA_DIR})"
(OUT_DIR / "shards").mkdir(parents=True, exist_ok=True)
torch.manual_seed(CFG["seed"])

In [3]:
logging.basicConfig(
    level=logging.WARNING, format="%(asctime)s | %(name)-5s | %(message)s", datefmt="%H:%M:%S", force=True,
    handlers=[logging.StreamHandler(sys.stdout), logging.FileHandler(OUT_DIR / "run.log")],
)
def get_logger(name):
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    return logger

log = get_logger("main")
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

def fmt_time(s):
    return f"{int(s // 60)}m{int(s % 60):02d}s"

log.info(f"config: {CFG}")

07:42:29 | main  | config: {'run_name': 'exp4_rft_data', 'model_id': 'Qwen/Qwen2.5-0.5B-Instruct', 'seed': 42, 'debug': False, 'n_samples': 8, 'temperature': 0.7, 'max_new_tokens': 512, 'prompts_per_batch': 64, 'flush_every': 10, 'max_kept_per_problem': 4, 'limit': None}


## Prompt + answer checking
`build_prompt` / `is_correct` are identical to 01. `extract_pred` is **stricter** than 01's on purpose (training data must be in a checkable format):
- number after the last `####`, else the number in the last `\boxed{...}`
- **no** "last number in the text" fallback
- returns `(pred, source)` with source `"####"`, `"boxed"` or `None`

In [4]:
INSTRUCTION = "Solve the problem step by step. End your answer with a final line of the form '#### <number>'."

def build_prompt(tok, question):
    msgs = [{"role": "user", "content": f"{question}\n\n{INSTRUCTION}"}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

NUM_RE = re.compile(r"-?\d[\d,]*(?:\.\d+)?")
BOXED_RE = re.compile(r"\\boxed\{([^{}]*)\}")

def extract_pred(text):
    if "####" in text:
        nums = NUM_RE.findall(text.split("####")[-1])
        if nums:
            return nums[0].replace(",", ""), "####"
    boxed = BOXED_RE.findall(text)
    if boxed:
        nums = NUM_RE.findall(boxed[-1])
        if nums:
            return nums[0].replace(",", ""), "boxed"
    return None, None

def is_correct(pred, gold):
    try:
        return pred is not None and abs(float(pred) - float(gold)) < 1e-6
    except ValueError:
        return False

assert extract_pred("so 1,000 total\n#### 1,000") == ("1000", "####")
assert extract_pred("#### -3") == ("-3", "####")
assert extract_pred("The answer is $\\boxed{18}$.") == ("18", "boxed")
assert extract_pred("\\boxed{2,500} so\n#### 2500") == ("2500", "####")    # both: #### wins
assert extract_pred("\\boxed{7} then \\boxed{9}") == ("9", "boxed")         # last box
assert extract_pred("3 + 4 = 7, so the answer is 72") == (None, None)      # no fallback
assert extract_pred("####") == (None, None)
assert is_correct("18.00", "18") and not is_correct("17", "18") and not is_correct(None, "18")

## Data: train + val 

In [5]:
data = {split: pd.read_json(DATA_DIR / f"gsm8k_{split}.jsonl", lines=True, dtype={"gold": str}) for split in ["train", "val"]}
if CFG["limit"]:
    data = {split: df.head(CFG["limit"]) for split, df in data.items()}
log.info(f"problems: { {split: len(df) for split, df in data.items()} }")

07:42:29 | main  | problems: {'train': 6973, 'val': 500}


## Model: base Qwen2.5-0.5B-Instruct, one copy per GPU

In [6]:
devices = [f"cuda:{i}" for i in range(torch.cuda.device_count())] or ["cpu"]
dtype = torch.float16 if devices[0].startswith("cuda") else torch.float32   # T4: no bf16
log.info(f"devices: {devices} | dtype: {dtype} | torch {torch.__version__} | transformers {transformers.__version__}")

tok = AutoTokenizer.from_pretrained(CFG["model_id"], padding_side="left")
models = []
for d in devices:
    t0 = time.time()
    models.append(AutoModelForCausalLM.from_pretrained(CFG["model_id"], dtype=dtype).to(d).eval())
    log.info(f"model loaded on {d} in {time.time() - t0:.1f}s")

07:42:29 | main  | devices: ['cuda:0', 'cuda:1'] | dtype: torch.float16 | torch 2.11.0+cu130 | transformers 5.20.2


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (3 outputs left out)

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

07:42:51 | main  | model loaded on cuda:0 in 19.8s


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

07:42:52 | main  | model loaded on cuda:1 in 1.1s


## Sampling
Prompts sorted by length (less padding), dealt round-robin to the GPUs, one thread per GPU.
`top_k=0, top_p=1, repetition_penalty=1` override Qwen's `generation_config.json` (top_k 20, top_p 0.8, rep. penalty 1.05) → plain T=0.7 sampling.

Resume: a shard keeps only problems with all `n_samples` rows (a crash can leave a half-written problem or line); those are skipped on rerun.

In [ ]:
@torch.inference_mode()
def sample_rank(rank, shard, split):
    rlog, model = get_logger(f"rank{rank}"), models[rank]
    bs, k = CFG["prompts_per_batch"], CFG["n_samples"]
    shard_path = OUT_DIR / "shards" / f"{split}_rank{rank}.jsonl"

    saved = []
    if shard_path.exists():
        for line in shard_path.read_text().splitlines():
            try:
                saved.append(json.loads(line))
            except json.JSONDecodeError:   # half-written last line
                pass
    done_ids = {pid for pid, n in Counter(r["problem_id"] for r in saved).items() if n == k}
    saved = [r for r in saved if r["problem_id"] in done_ids]
    
    shard_path.write_text("".join(json.dumps(r) + "\n" for r in saved))   # drop incomplete problems
    todo = [item for item in shard if item[0] not in done_ids]
    n_batches = (len(todo) + bs - 1) // bs
    rlog.info(f"{split} | {len(shard)} problems, {len(done_ids)} already saved, {len(todo)} to do in {n_batches} batches")

    buffer, n_correct, n_seen, t0 = [], sum(r["correct"] for r in saved), len(saved), time.time()
    for b in range(n_batches):
        chunk, tb = todo[b * bs:(b + 1) * bs], time.time()
        enc = tok([prompt for _, prompt, _, _ in chunk], return_tensors="pt", padding=True).to(model.device)
        out = model.generate(**enc, do_sample=True, temperature=CFG["temperature"], top_k=0, top_p=1.0,
                             repetition_penalty=1.0, num_return_sequences=k,
                             max_new_tokens=CFG["max_new_tokens"], pad_token_id=tok.pad_token_id)
        new = out[:, enc.input_ids.shape[1]:]          # [len(chunk) * k, L]; rows i*k .. i*k+k-1 belong to prompt i
        n_tokens = (new != tok.pad_token_id).sum(dim=1).tolist()
        for row, (text, n) in enumerate(zip(tok.batch_decode(new, skip_special_tokens=True), n_tokens)):
            problem_id, _, question, gold = chunk[row // k]
            pred, source = extract_pred(text)
            ok = is_correct(pred, gold)
            buffer.append(dict(problem_id=problem_id, sample_id=row % k, question=question, gold=gold, response=text,
                               pred=pred, source=source, correct=ok, n_tokens=n, truncated=n >= CFG["max_new_tokens"]))
            n_correct, n_seen = n_correct + ok, n_seen + 1

        if (b + 1) % CFG["flush_every"] == 0 or b == n_batches - 1:
            with open(shard_path, "a") as f:
                f.write("".join(json.dumps(r) + "\n" for r in buffer))
            buffer = []
        elapsed, problems_done = time.time() - t0, len(done_ids) + min((b + 1) * bs, len(todo))
        rlog.info(f"{split} | batch {b + 1}/{n_batches} | {problems_done}/{len(shard)} problems | "
                  f"sample acc so far {n_correct / n_seen:.3f} | {sum(n_tokens) / (time.time() - tb):.0f} tok/s | "
                  f"elapsed {fmt_time(elapsed)} | ETA {fmt_time(elapsed / (b + 1) * (n_batches - b - 1))}")

In [8]:
samples, wall = {}, {}
for split, df in data.items():
    prompts = [build_prompt(tok, q) for q in df.question]
    order = sorted(range(len(df)), key=lambda i: len(prompts[i]), reverse=True)
    shards = [[(i, prompts[i], df.question[i], df.gold[i]) for i in order[r::len(models)]] for r in range(len(models))]
    t0 = time.time()
    with ThreadPoolExecutor(len(models)) as pool:
        list(pool.map(sample_rank, range(len(models)), shards, [split] * len(models)))   # list() re-raises thread errors
    wall[split] = time.time() - t0

    rows = [json.loads(line) for r in range(len(models))
            for line in (OUT_DIR / "shards" / f"{split}_rank{r}.jsonl").read_text().splitlines()]
    samples[split] = pd.DataFrame(rows).sort_values(["problem_id", "sample_id"]).reset_index(drop=True)
    assert len(samples[split]) == len(df) * CFG["n_samples"], f"{split}: expected {len(df) * CFG['n_samples']} samples"
    samples[split].to_json(OUT_DIR / f"samples_{split}.jsonl", orient="records", lines=True)
    log.info(f"{split} | {len(samples[split])} samples in {fmt_time(wall[split])} (this run) | "
             f"sample acc {samples[split].correct.mean():.3f}")

07:42:52 | rank1 | train | 3486 problems, 0 already saved, 3486 to do in 55 batches
07:42:52 | rank0 | train | 3487 problems, 0 already saved, 3487 to do in 55 batches
07:46:51 | rank0 | train | batch 1/55 | 64/3487 problems | sample acc so far 0.227 | 767 tok/s | elapsed 3m58s | ETA 214m34s
07:46:51 | rank1 | train | batch 1/55 | 64/3486 problems | sample acc so far 0.197 | 769 tok/s | elapsed 3m58s | ETA 214m50s
07:50:26 | rank0 | train | batch 2/55 | 128/3487 problems | sample acc so far 0.217 | 824 tok/s | elapsed 7m33s | ETA 200m24s
07:50:40 | rank1 | train | batch 2/55 | 128/3486 problems | sample acc so far 0.196 | 827 tok/s | elapsed 7m47s | ETA 206m40s
07:53:54 | rank0 | train | batch 3/55 | 192/3487 problems | sample acc so far 0.229 | 851 tok/s | elapsed 11m01s | ETA 191m11s
07:54:26 | rank1 | train | batch 3/55 | 192/3486 problems | sample acc so far 0.215 | 794 tok/s | elapsed 11m33s | ETA 200m19s
07:57:22 | rank0 | train | batch 4/55 | 256/3487 problems | sample acc so fa

## Filter + dedupe → RFT data
- keep: correct and not truncated
- if the last line isn't a `####` line (e.g. answered with `\boxed{}`), append `\n#### {gold}`; nothing else in the text changes
- dedupe per problem by the **sequence of equations** (`a op b = c`, spaces / `$` / commas removed, `×`,`x`→`*`, `÷`→`/`);
  an answer with no equations falls back to its whitespace-normalized text. Keep the first `max_kept_per_problem` unique ones.

In [9]:
NUMBER = r"\(?\$?\d[\d,]*(?:\.\d+)?%?\)?"
EQ_RE = re.compile(rf"({NUMBER}(?:\s*[-+*/×÷x]\s*{NUMBER})+)\s*=\s*\$?(-?\d[\d,]*(?:\.\d+)?)")
OP_MAP = str.maketrans({"×": "*", "x": "*", "÷": "/"})

assert EQ_RE.findall("$2000-$600 = $1400") == [("$2000-$600", "1400")]
assert EQ_RE.findall("3 x 4 = 12 and 12 ÷ 2 = 6.") == [("3 x 4", "12"), ("12 ÷ 2", "6")]
assert EQ_RE.findall("20% * 500 = 100") == [("20% * 500", "100")]

rft, problem_stats = {}, {}
for split, df in samples.items():
    kept_rows, n_kept = [], Counter()
    for problem_id, group in df[df.correct & ~df.truncated].groupby("problem_id"):
        seen = set()
        for r in group.itertuples():
            equations = tuple(re.sub(r"[\s$,]", "", lhs).translate(OP_MAP) + "=" + rhs.replace(",", "").rstrip(".")
                              for lhs, rhs in EQ_RE.findall(r.response))
            key = equations or " ".join(r.response.split())
            if key in seen or n_kept[problem_id] >= CFG["max_kept_per_problem"]:
                continue
            seen.add(key)
            n_kept[problem_id] += 1
            last_line = r.response.rstrip().split("\n")[-1].strip()
            solution = r.response if last_line.startswith("####") else r.response.rstrip() + f"\n#### {r.gold}"
            kept_rows.append(dict(question=r.question, solution=solution, gold=r.gold))
    rft[split] = pd.DataFrame(kept_rows, columns=["question", "solution", "gold"])
    rft[split].to_json(OUT_DIR / f"rft_{split}.jsonl", orient="records", lines=True)

    problem_stats[split] = (df.groupby("problem_id").correct.sum().rename("n_correct").to_frame()
                            .assign(n_kept=lambda s: [n_kept[pid] for pid in s.index]))
    problem_stats[split].to_csv(OUT_DIR / f"problem_stats_{split}.csv")
    log.info(f"{split} | kept {len(rft[split])} answers for {(problem_stats[split].n_kept > 0).sum()}/{len(problem_stats[split])} problems")

print(rft["train"].solution.iloc[0] if len(rft["train"]) else "nothing kept")

10:58:12 | main  | train | kept 16515 answers for 5982/6973 problems
10:58:13 | main  | val | kept 1209 answers for 436/500 problems
To determine how much money Mr. Ben is remaining with, we need to follow the sequence of events step by step.

1. **Initial Amount**: Mr. Ben starts with $2000.
2. **Payment from Supplier**:
   - Mr. Ben writes a cheque for $600 to the supplier.
   - After receiving the cheque, Mr. Ben has:
     \[
     2000 - 600 = 1400
     \]
3. **Payment from Debtor**:
   - Mr. Ben's debtor pays him $800 from the purchases they had made on credit.
   - After receiving the payment, Mr. Ben has:
     \[
     1400 + 800 = 2200
     \]
4. **Cost of Equipment Maintenance**:
   - Mr. Ben spends $1200 on the whole operation.
   - After the maintenance, Mr. Ben has:
     \[
     2200 - 1200 = 1000
     \]

Thus, the final amount of money Mr. Ben is remaining with is \boxed{1000}.
#### 1000


## Metrics + save

In [10]:
n = CFG["n_samples"]
metrics = {}
for split, df in samples.items():
    stats, kept = problem_stats[split], rft[split]
    gold_tokens = data[split].solution.map(lambda s: len(tok(s).input_ids))
    kept_tokens = kept.solution.map(lambda s: len(tok(s).input_ids))
    metrics[split] = dict(
        n_problems=len(stats),
        coverage=round(float((stats.n_correct > 0).mean()), 4),                    # >= 1 correct of n
        pass_at_1=round(float(df.correct.mean()), 4),
        **{f"pass_at_{n}": round(float((stats.n_correct > 0).mean()), 4)},      # = coverage when k = n
        truncated_rate=round(float(df.truncated.mean()), 4),
        n_kept=len(kept),
        mean_kept_per_problem=round(float(stats.n_kept.mean()), 3),
        median_tokens_kept=float(kept_tokens.median()) if len(kept) else None,
        median_tokens_gold=float(gold_tokens.median()),
        source_all=df.source.fillna("none").value_counts(normalize=True).round(4).to_dict(),
        source_correct=df[df.correct].source.value_counts(normalize=True).round(4).to_dict(),
        wall_min_this_run=round(wall[split] / 60, 2),
        gpu_hours_this_run=round(wall[split] * len(models) / 3600, 3) if devices[0].startswith("cuda") else 0.0,
    )
(OUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))
(OUT_DIR / "config.json").write_text(json.dumps(
    {**CFG, "instruction": INSTRUCTION, "devices": devices, "dtype": str(dtype),
     "torch": torch.__version__, "transformers": transformers.__version__}, indent=2))
for p in sorted(OUT_DIR.glob("*")):
    if p.is_file():
        log.info(f"saved {p.name} ({p.stat().st_size / 1e6:.1f} MB)")
pd.DataFrame(metrics)

10:58:27 | main  | saved config.json (0.0 MB)
10:58:27 | main  | saved metrics.json (0.0 MB)
10:58:27 | main  | saved problem_stats_train.csv (0.1 MB)
10:58:27 | main  | saved problem_stats_val.csv (0.0 MB)
10:58:27 | main  | saved rft_train.jsonl (19.9 MB)
10:58:27 | main  | saved rft_val.jsonl (1.4 MB)
10:58:27 | main  | saved run.log (0.0 MB)
10:58:27 | main  | saved samples_train.jsonl (77.8 MB)
10:58:27 | main  | saved samples_val.jsonl (5.5 MB)


,train,val
n_problems,6973,500
coverage,0.8595,0.872
pass_at_1,0.4997,0.5185
pass_at_8,0.8595,0.872
truncated_rate,0.0498,0.0428
n_kept,16515,1209
mean_kept_per_problem,2.368,2.418
median_tokens_kept,267.0,259.0
median_tokens_gold,82.0,78.0
source_all,"{'boxed': 0.547, '####': 0.3353, 'none': 0.1177}","{'boxed': 0.5445, '####': 0.343, 'none': 0.1125}"
